In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# Avaliação de Agentes

Agentes de IA são **não-determinísticos** por natureza. A mesma pergunta pode gerar trajetórias diferentes, chamadas de tools em ordens distintas, e respostas com variações de texto. Isso torna testes tradicionais (tipo `assert resposta == "valor esperado"`) insuficientes.

Pra lidar com isso, a gente precisa de **evals** -- avaliações sistemáticas que medem se o agente está se comportando como esperado, mesmo quando as respostas variam. Nessa aula, vamos explorar as principais técnicas: matching de trajetória, LLM-as-Judge, detecção de alucinações e test suites.

In [2]:
import json
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage

## Construindo o agente

Antes de avaliar, a gente precisa de um agente pra testar. Vamos criar um assistente meteorológico simples com duas tools: uma que consulta o clima de uma cidade e outra que converte temperatura de Celsius pra Fahrenheit.

É um agente propositalmente simples -- o foco aqui não é o agente em si, mas sim como a gente avalia o comportamento dele de forma sistemática.

In [3]:
@tool
def consultar_clima(cidade: str) -> str:
    """Consulta a previsão do tempo para uma cidade."""
    previsoes = {
        "são paulo": "25°C, parcialmente nublado",
        "rio de janeiro": "32°C, ensolarado",
        "curitiba": "15°C, chuva leve",
    }
    return previsoes.get(cidade.lower(), f"Previsão não disponível para {cidade}.")


@tool
def converter_temperatura(celsius: float) -> float:
    """Converte temperatura de Celsius para Fahrenheit."""
    return round(celsius * 9 / 5 + 32, 1)

In [4]:
agente = create_agent(
    model="gpt-4.1-nano",
    tools=[consultar_clima, converter_temperatura],
    system_prompt="Você é um assistente meteorológico.",
)

In [5]:
resposta = agente.invoke(
    {"messages": [HumanMessage(content="Qual o clima em São Paulo?")]}
)

for msg in resposta["messages"]:
    print(f"{msg.type}: {msg.content[:200]}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  tool_calls: {[tc['name'] for tc in msg.tool_calls]}")

human: Qual o clima em São Paulo?
ai: 
  tool_calls: ['consultar_clima']
tool: 25°C, parcialmente nublado
ai: O clima em São Paulo está parcialmente nublado com temperatura de aproximadamente 25°C.


## Trajetória do agente

Quando um agente roda, ele produz uma sequência de passos: recebe a pergunta, decide qual tool chamar, recebe o resultado, formula a resposta. Essa sequência é a **trajetória** do agente.

Avaliar a trajetória é fundamental porque a gente quer garantir que o agente está tomando as decisões certas, não só que a resposta final parece ok. Vamos extrair a trajetória das mensagens retornadas.

In [6]:
# As mensagens do agente já estão no formato aceito pelo agentevals
trajetoria = resposta["messages"]

for msg in trajetoria:
    print(f"{msg.type}: {msg.content[:100]}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  tools: {[tc['name'] for tc in msg.tool_calls]}")

human: Qual o clima em São Paulo?
ai: 
  tools: ['consultar_clima']
tool: 25°C, parcialmente nublado
ai: O clima em São Paulo está parcialmente nublado com temperatura de aproximadamente 25°C.


## Avaliação por matching de trajetória

A forma mais direta de avaliar um agente é comparar a **trajetória real** com uma **trajetória de referência**. Isso é puramente determinístico -- não precisa de LLM, não custa nada, é rápido.

O `agentevals` oferece quatro modos de comparação: **strict** (ordem e argumentos exatos), **unordered** (mesmas tools, qualquer ordem), **subset** (referência contida na real) e **superset** (real contida na referência). Vamos começar definindo a trajetória de referência.

In [7]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

# Trajetória de referência no formato OpenAI (dicts)
referencia = [
    {"role": "user", "content": "Qual o clima em São Paulo?"},
    {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {"function": {"name": "consultar_clima", "arguments": json.dumps({"cidade": "São Paulo"})}}
        ],
    },
    {"role": "tool", "content": "25°C, parcialmente nublado"},
    {"role": "assistant", "content": "O clima em São Paulo é 25°C, parcialmente nublado."},
]

In [8]:
evaluator_strict = create_trajectory_match_evaluator(trajectory_match_mode="strict")

resultado = evaluator_strict(outputs=trajetoria, reference_outputs=referencia)
print(resultado)

{'key': 'trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}


O modo **strict** compara tool por tool, na ordem exata, incluindo os argumentos. Se tudo bater, o score é `True`. Vamos testar também os modos **unordered** (ignora a ordem das tools) e **subset** (verifica se a referência é um subconjunto).

In [9]:
evaluator_unordered = create_trajectory_match_evaluator(trajectory_match_mode="unordered")

resultado = evaluator_unordered(outputs=trajetoria, reference_outputs=referencia)
print(resultado)

{'key': 'trajectory_unordered_match', 'score': True, 'comment': None, 'metadata': None}


In [10]:
evaluator_subset = create_trajectory_match_evaluator(trajectory_match_mode="subset")

resultado = evaluator_subset(outputs=trajetoria, reference_outputs=referencia)
print(resultado)

{'key': 'trajectory_subset_match', 'score': True, 'comment': None, 'metadata': None}


Às vezes os argumentos da tool podem ter diferenças de capitalização -- "São Paulo" vs "são paulo". Com `tool_args_match_overrides`, a gente define uma função customizada pra comparar os argumentos de cada tool.

In [11]:
# Matching customizado: case-insensitive para argumentos
evaluator_custom = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_overrides={
        "consultar_clima": lambda x, y: x["cidade"].lower() == y["cidade"].lower()
    },
)

resultado = evaluator_custom(outputs=trajetoria, reference_outputs=referencia)
print(resultado)

{'key': 'trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}


## LLM como juiz

O matching determinístico é rápido e barato, mas é rígido. Se o agente chamar a tool certa com argumentos ligeiramente diferentes, o teste falha.

A alternativa é usar um **LLM como juiz**. Em vez de comparar strings, a gente pede pra um modelo avaliar se a trajetória faz sentido. Isso custa chamadas de API, mas é muito mais flexível.

O `agentevals` traz prompts prontos pra isso: um que avalia a trajetória **sem referência** (só pelo bom senso) e outro que compara **com uma referência**.

In [12]:
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT,
    TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,
)

In [13]:
# LLM avalia a trajetória SEM referência
evaluator_llm = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_ACCURACY_PROMPT,
    model="openai:gpt-4.1-mini",
)

resultado = evaluator_llm(outputs=trajetoria)
print(resultado)

{'key': 'trajectory_accuracy', 'score': True, 'comment': "The user's goal is to know the current weather in São Paulo. The AI assistant correctly identifies this goal and makes a logical step by calling a weather consultation tool with the argument 'São Paulo.' The response from the tool provides the weather information: 25°C and partly cloudy, which the assistant then clearly communicates back to the user. The trajectory is logically consistent, shows clear progression from question to answer, and is efficient without any unnecessary steps. Thus, the score should be: true.", 'metadata': None}


Olha só o reasoning que o LLM retornou. Ele analisou cada passo da trajetória e julgou se fazia sentido. Essa é a grande vantagem do LLM-as-Judge: ele entende **contexto e intenção**, não só compara strings.

Agora vamos testar com uma trajetória de referência, pra ver se o LLM consegue comparar as duas.

In [14]:
# LLM avalia COM referência
evaluator_llm_ref = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,
    model="openai:gpt-4.1-mini",
)

resultado = evaluator_llm_ref(outputs=trajetoria, reference_outputs=referencia)
print(resultado)

{'key': 'trajectory_accuracy', 'score': True, 'comment': 'The actual trajectory logically follows the steps of the reference trajectory: from the user query, to a tool call querying the weather in São Paulo, receiving the tool result with the temperature and condition, and a final assistant utterance reporting the weather. The progression is clear and efficient, mirroring the reference path with additional detail in phrasing but conveying the same semantic information — temperature around 25°C and partly cloudy skies. Therefore, the actual trajectory is semantically equivalent to the reference, makes logical sense, shows clear progression, and is reasonably efficient. Thus, the score should be: true.', 'metadata': None}


Em vez de True/False, às vezes a gente quer uma **nota contínua** entre 0.0 e 1.0. Basta passar `continuous=True` na criação do evaluator.

In [15]:
# Nota contínua (0.0 a 1.0) em vez de True/False
evaluator_continuo = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_ACCURACY_PROMPT,
    model="openai:gpt-4.1-mini",
    continuous=True,
)

resultado = evaluator_continuo(outputs=trajetoria)
print(resultado)

{'key': 'trajectory_accuracy', 'score': 1.0, 'comment': "The trajectory's goal is to answer the user's question about the current weather in São Paulo. The steps within the trajectory are logically consistent: the assistant correctly identifies the user's query, calls the appropriate weather-consulting tool with the correct city parameter, receives the weather information, and provides a clear and accurate final response to the user. The progression is clear and efficient, with no unnecessary steps. This satisfies all rubric criteria—logical sense between steps, clear progression, and relative efficiency. Thus, the score should be: 1.0.", 'metadata': None}


## Avaliação da resposta final

Até agora a gente avaliou a **trajetória** -- as ferramentas que o agente escolheu e a ordem em que chamou. Mas isso não diz nada sobre a qualidade da resposta final que o usuário recebeu.

Pra isso, a gente usa o **openevals**, que vem com prompts prontos como o `CORRECTNESS_PROMPT`. Ele compara a resposta do agente com uma referência e diz se o conteúdo está correto.

In [16]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

In [17]:
# Avaliação de corretude da resposta final
evaluator_corretude = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="openai:gpt-4.1-mini",
)

resultado = evaluator_corretude(
    inputs="Qual o clima em São Paulo?",
    outputs=resposta["messages"][-1].content,
    reference_outputs="O clima em São Paulo está 25°C, parcialmente nublado.",
)
print(resultado)

{'key': 'score', 'score': True, 'comment': "The question asks about the climate in São Paulo. The model's output states that the climate is partially cloudy with a temperature of about 25°C. This matches the information given in the reference output, which also mentions a temperature of 25°C and partial cloudiness. There are no factual errors or misleading statements. The terminology used is accurate and appropriate. The answer addresses the question fully by describing both temperature and sky condition, both key aspects of climate. Therefore, the answer is accurate, complete, logically consistent, and uses precise terminology. Thus, the score should be: true.", 'metadata': None}


## Detector de alucinações

Uma das avaliações mais úteis na prática é detectar **alucinações** -- quando o agente inventa informações que não estão nos dados reais. Pra isso, a gente cria um prompt customizado que compara a resposta com o contexto original.

Vamos testar com uma resposta que contém uma informação inventada: "ventos de 30km/h".

In [18]:
PROMPT_ALUCINACAO = """Avalie se a resposta contém informações que NÃO estão presentes no contexto fornecido.
Se a resposta contiver informações inventadas, é uma alucinação.

<context>{context}</context>
<input>{inputs}</input>
<output>{outputs}</output>

Responda True se a resposta é fiel ao contexto, False se contém alucinações."""

evaluator_alucinacao = create_llm_as_judge(
    prompt=PROMPT_ALUCINACAO,
    model="openai:gpt-4.1-mini",
)

resultado = evaluator_alucinacao(
    inputs="Qual o clima em São Paulo?",
    outputs="O clima em São Paulo está 25°C, parcialmente nublado, com ventos de 30km/h.",
    context="São Paulo: 25°C, parcialmente nublado.",
)
print(resultado)

{'key': 'score', 'score': False, 'comment': 'O contexto informa que em São Paulo está 25°C e parcialmente nublado. A resposta inclui essa informação, mas também menciona "ventos de 30km/h", que não está presente no contexto fornecido. Portanto, a resposta contém informações que não estão no contexto, caracterizando uma alucinação. Assim, a pontuação deve ser: False.', 'metadata': None}


Repare que o score deu `False` -- e isso é o comportamento correto! A resposta que a gente passou continha "ventos de 30km/h", uma informação que não existe no contexto original. O evaluator detectou a **alucinação** com sucesso.

## Avaliação de grafos LangGraph

Quando o agente é construído como um **grafo LangGraph**, a gente tem acesso a uma informação extra: a sequência de **nós** que o grafo visitou durante a execução. Isso é diferente da trajetória de mensagens -- aqui a gente vê o fluxo estrutural do grafo.

Pra extrair essa informação, o agente precisa de um **checkpointer** que salva o estado a cada passo. Vamos recriar nosso agente com `MemorySaver` e rodar com um `thread_id`.

In [19]:
from langgraph.checkpoint.memory import MemorySaver
from agentevals.graph_trajectory.utils import extract_langgraph_trajectory_from_thread

In [20]:
checkpointer = MemorySaver()

agente_rastreado = create_agent(
    model="gpt-4.1-nano",
    tools=[consultar_clima, converter_temperatura],
    system_prompt="Você é um assistente meteorológico.",
    checkpointer=checkpointer,
)

In [21]:
config = {"configurable": {"thread_id": "eval-thread-1"}}

resposta = agente_rastreado.invoke(
    {"messages": [HumanMessage(content="Qual o clima em Curitiba? Converta para Fahrenheit.")]},
    config=config,
)

print(resposta["messages"][-1].content)

O clima em Curitiba está com uma temperatura de 15°C, com chuva leve. Convertendo para Fahrenheit, a temperatura é aproximadamente 59°F.


Com o `thread_id`, o LangGraph salva cada passo no checkpointer. A função `extract_langgraph_trajectory_from_thread` recupera a sequência de nós visitados no grafo — por exemplo, `['__start__', 'model', 'tools', 'tools', 'model', 'tools', 'model']` quando o agente chama múltiplas ferramentas.

In [22]:
trajetoria_grafo = extract_langgraph_trajectory_from_thread(
    agente_rastreado, config
)

print("Inputs:", trajetoria_grafo["inputs"])
print("Steps:", trajetoria_grafo["outputs"]["steps"])
print("Num results:", len(trajetoria_grafo["outputs"]["results"]))

Inputs: [{'__start__': {'messages': [HumanMessage(content='Qual o clima em Curitiba? Converta para Fahrenheit.', additional_kwargs={}, response_metadata={}, id='7729911b-7cdd-4e09-8ab8-737ca382c292')]}}]
Steps: [['__start__', 'model', 'tools', 'model', 'tools', 'model']]
Num results: 1


Agora a gente pode avaliar essa trajetória de grafo com um LLM judge, e também verificar se a sequência de nós visitados bate com o esperado.

In [23]:
# Avaliar as mensagens do agente com LLM judge
# Usamos a trajetória de mensagens (não o formato de grafo) para compatibilidade
from agentevals.trajectory.llm import create_trajectory_llm_as_judge, TRAJECTORY_ACCURACY_PROMPT

graph_llm_evaluator = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_ACCURACY_PROMPT,
    model="openai:gpt-4.1-mini",
)

# Extrair mensagens do último resultado
mensagens_grafo = trajetoria_grafo["outputs"]["results"][-1].get("messages", [])
resultado = graph_llm_evaluator(outputs=mensagens_grafo)
print(resultado)
print(f"\nSteps do grafo: {trajetoria_grafo['outputs']['steps']}")

{'key': 'trajectory_accuracy', 'score': True, 'comment': 'The trajectory consists of a single step where the assistant provides the current weather condition in Curitiba with a temperature of 15°C and light rain, then converts the temperature to Fahrenheit as approximately 59°F. Assuming the goal is to report the current weather in Curitiba including a temperature conversion from Celsius to Fahrenheit, the trajectory makes logical sense, shows clear progression (weather statement and conversion), and is efficient (done in one step without unnecessary detail). Thus, the score should be: true.', 'metadata': None}

Steps do grafo: [['__start__', 'model', 'tools', 'model', 'tools', 'model']]


Também dá pra fazer **strict match** nos steps do grafo, comparando a sequência exata de nós visitados com uma referência.

In [24]:
from agentevals.graph_trajectory.strict import graph_trajectory_strict_match

referencia_grafo = {
    "results": [],
    "steps": trajetoria_grafo["outputs"]["steps"],
}

resultado = graph_trajectory_strict_match(
    outputs=trajetoria_grafo["outputs"],
    reference_outputs=referencia_grafo,
)
print(resultado)

{'key': 'graph_trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}


**Nota:** Aqui estamos comparando a trajetória com ela mesma — na prática, você definiria os steps esperados de forma independente.

## Montando um test suite

Testar um caso isolado é útil pra debugar, mas pra ter confiança de verdade a gente precisa de um **test suite** -- um conjunto de cenários que cobre os principais comportamentos esperados do agente.

A ideia é simples: definir entrada, tool esperada e resposta de referência pra cada caso. Depois rodar tudo de uma vez e ver quantos passam.

In [25]:
# Mini test suite: vários cenários de teste
casos_teste = [
    {
        "input": "Qual o clima em São Paulo?",
        "tool_esperada": "consultar_clima",
        "referencia": "25°C, parcialmente nublado",
    },
    {
        "input": "Qual o clima no Rio de Janeiro?",
        "tool_esperada": "consultar_clima",
        "referencia": "32°C, ensolarado",
    },
    {
        "input": "Converta 25 graus para Fahrenheit.",
        "tool_esperada": "converter_temperatura",
        "referencia": "77.0°F",
    },
]

In [26]:
evaluator_corretude = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="openai:gpt-4.1-mini",
)

resultados = []

for i, caso in enumerate(casos_teste):
    resposta = agente.invoke(
        {"messages": [HumanMessage(content=caso["input"])]}
    )

    # Verificar se a tool correta foi chamada
    tools_chamadas = []
    for msg in resposta["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tools_chamadas.extend([tc["name"] for tc in msg.tool_calls])

    tool_correta = caso["tool_esperada"] in tools_chamadas

    # Avaliar corretude da resposta
    eval_result = evaluator_corretude(
        inputs=caso["input"],
        outputs=resposta["messages"][-1].content,
        reference_outputs=caso["referencia"],
    )

    resultados.append({
        "caso": caso["input"],
        "tool_correta": tool_correta,
        "corretude": eval_result["score"],
    })

    print(f"Caso {i+1}: tool={'OK' if tool_correta else 'FALHA'}, corretude={eval_result['score']}")

Caso 1: tool=OK, corretude=True
Caso 2: tool=OK, corretude=False
Caso 3: tool=OK, corretude=True


In [27]:
# Resumo do test suite
total = len(resultados)
tools_ok = sum(1 for r in resultados if r["tool_correta"])
corretude_ok = sum(1 for r in resultados if r["corretude"])

print(f"Tools corretas: {tools_ok}/{total}")
print(f"Respostas corretas: {corretude_ok}/{total}")

Tools corretas: 3/3
Respostas corretas: 2/3


Com isso a gente fecha o ciclo completo de avaliação de agentes. Vimos como testar a **trajetória** (as ferramentas que o agente escolheu), a **resposta final** (se o conteúdo está correto) e como detectar **alucinações** (informações inventadas).

Na prática, você vai querer montar um test suite como esse último e rodar toda vez que mudar o prompt, trocar o modelo ou adicionar ferramentas novas. Eval não é algo que a gente faz uma vez e esquece -- é parte do ciclo de desenvolvimento do agente.